In [16]:
import time
import tiktoken
import torch
import torch.nn as nn

In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
            persistent=False
        )

        ####################################################
        # NEW
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.ptr_current_pos = 0
        ####################################################

    def forward(self, x, use_cache=False):
        b, num_tokens, d_in = x.shape

        keys_new = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        values_new = self.W_value(x)
        queries = self.W_query(x)

        print("MHA with KV")
        print("queries : ", queries.shape)
        print("keys new : ", keys_new.shape)
        print("values new : ", values_new.shape)


        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys_new = keys_new.view(b, num_tokens, self.num_heads, self.head_dim)
        values_new = values_new.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        ####################################################
        # NEW
        if use_cache:
            if self.cache_k is None:
                self.cache_k, self.cache_v = keys_new, values_new
            else:
                self.cache_k = torch.cat([self.cache_k, keys_new], dim=1)
                self.cache_v = torch.cat([self.cache_v, values_new], dim=1)
            keys, values = self.cache_k, self.cache_v
        else:
            keys, values = keys_new, values_new
        ####################################################

        print("cached keys  : ", keys.shape)
        print("cached values : ", values.shape)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        print("attn scores : ", attn_scores.shape)

        ####################################################
        # NEW
        num_tokens_Q = queries.shape[-2]
        num_tokens_K = keys.shape[-2]
        if use_cache:
            mask_bool = self.mask.bool()[
                self.ptr_current_pos:self.ptr_current_pos + num_tokens_Q, :num_tokens_K
            ]
            self.ptr_current_pos += num_tokens_Q
        ####################################################
        # Original mask truncated to the number of tokens and converted to boolean
        else:
            mask_bool = self.mask.bool()[:num_tokens_Q, :num_tokens_K]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        print("context_vec: ", context_vec.shape)

        return context_vec

    ####################################################
    # NEW
    def reset_cache(self):
        self.cache_k, self.cache_v = None, None
        self.ptr_current_pos = 0
    ####################################################


The reset_cache() function is clearing the Cache.

We are reseting both the keys and value buffers between two separate text-generation calls (before giving the next prompt).

In [18]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


In [19]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, use_cache=False):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)

        # x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        ####################################################
        # NEW
        x = self.att(x, use_cache=use_cache)
        ####################################################

        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

In [20]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # self.trf_blocks = nn.Sequential(
        #    *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        ####################################################
        # NEW
        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.current_pos = 0
        ####################################################

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx, use_cache=False):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        # pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        ####################################################
        # NEW

        if use_cache:
            pos_ids = torch.arange(self.current_pos, self.current_pos + seq_len, device=in_idx.device, dtype=torch.long)
            self.current_pos += seq_len
        else:
            pos_ids = torch.arange(0, seq_len, device=in_idx.device, dtype=torch.long)
        pos_embeds = self.pos_emb(pos_ids).unsqueeze(0)
        ####################################################

        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)

        # x = self.trf_blocks(x)
        ####################################################
        # NEW
        for blk in self.trf_blocks:
            x = blk(x, use_cache=use_cache)
        ####################################################

        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

    ####################################################
    # NEW
    def reset_kv_cache(self):
        for blk in self.trf_blocks:
            blk.att.reset_cache()
        self.current_pos = 0
    ####################################################

In [21]:
GPT_CONFIG_124M = {
        "vocab_size": 50257,     # Vocabulary size
        "context_length": 1024,  # Context length
        "emb_dim": 768,          # Embedding dimension
        "n_heads": 1,           # Number of attention heads
        "n_layers": 1,          # Number of layers
        "drop_rate": 0.1,        # Dropout rate
        "qkv_bias": False        # Query-Key-Value bias
    }

Note: for understading purpose I have taken, 

n_layers = 1 and n_head = 1 in GPT_CONFIG_124M

we will able to illustrate better when we will see the output of the generate_text_simple_cached() function

In [22]:
model = GPTModel(GPT_CONFIG_124M)

In [23]:
# This was our function that we used when we are not using kv cache 

def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

Below is the new modified generate_text_simple_cached function for kv cached

In [24]:
def generate_text_simple_cached(model, idx, max_new_tokens, context_size=None, use_cache=True):
  model.eval()

  ctx_len = context_size or model.pos_emb.num_embeddings

  with torch.no_grad():
    if use_cache:
      # Init cache with full prompt
      model.reset_kv_cache()

      logits = model(idx[:, -ctx_len:], use_cache=True)
      
      print(logits.shape)
      
      print()
      print("inputs prompt is Cached is done")
      print()

      for i in range(max_new_tokens):

        # a) pick the token with the highest log-proability (greedy sampling)
            # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]
        # get the idx 
        next_idx = torch.argmax(logits, dim=1, keepdim=True)   # (batch, 1)

        # b) Append sampled index to the running sequence
        idx = torch.cat((idx, next_idx), dim=1)  # (batch, n_tokens+1)

        # c) feed the model only new token and not the whole concatenated idx
        logits = model(next_idx, use_cache=True)

        print(i+1, logits.shape)
    else:
      for _ in range(max_new_tokens):
        logits = model(idx[:, -ctx_len:], use_cache=False)

        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # get the idx 
        next_idx = torch.argmax(logits, dim=1, keepdim=True)   # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, next_idx), dim=1)  # (batch, n_tokens+1)
    return idx

In [28]:
tokenizer = tiktoken.get_encoding('gpt2')

In [29]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: tensor([[15496,    11,   314,   716]])


In [30]:
model.eval() 
model = GPTModel(GPT_CONFIG_124M)
max_new_tokens = 6  # we have max new token to generete is 6

out = generate_text_simple_cached(model=model, idx=encoded_tensor, max_new_tokens=6, context_size=GPT_CONFIG_124M['context_length'], use_cache=True)

print("Output:", out)
print("Output length:", len(out[0]))

MHA with KV
queries :  torch.Size([1, 4, 768])
keys new :  torch.Size([1, 4, 768])
values new :  torch.Size([1, 4, 768])
cached keys  :  torch.Size([1, 4, 1, 768])
cached values :  torch.Size([1, 4, 1, 768])
attn scores :  torch.Size([1, 1, 4, 4])
context_vec:  torch.Size([1, 4, 768])
torch.Size([1, 4, 50257])

inputs prompt is Cached is done

MHA with KV
queries :  torch.Size([1, 1, 768])
keys new :  torch.Size([1, 1, 768])
values new :  torch.Size([1, 1, 768])
cached keys  :  torch.Size([1, 5, 1, 768])
cached values :  torch.Size([1, 5, 1, 768])
attn scores :  torch.Size([1, 1, 1, 5])
context_vec:  torch.Size([1, 1, 768])
1 torch.Size([1, 1, 50257])
MHA with KV
queries :  torch.Size([1, 1, 768])
keys new :  torch.Size([1, 1, 768])
values new :  torch.Size([1, 1, 768])
cached keys  :  torch.Size([1, 6, 1, 768])
cached values :  torch.Size([1, 6, 1, 768])
attn scores :  torch.Size([1, 1, 1, 6])
context_vec:  torch.Size([1, 1, 768])
2 torch.Size([1, 1, 50257])
MHA with KV
queries :  tor

Explanation

As we can see at first it is storing the full input prompt, then we are passing the only last generated token to the model 

model(input_prompt)

queries = [1, 4, 768]  # (b, num_tokens, d_out)

keys = [1, 4, 768]  # (b, num_tokens, d_out)

values = [1, 4, 768]  # (b, num_tokens, d_out)


cached keys  :  torch.Size([1, 4, 1, 768])  (b, num_tokens, num_head, head_dim)

cached values :  torch.Size([1, 4, 1, 768])  (b, num_tokens, num_head, head_dim)

attn scores :  torch.Size([1, 1, 4, 4])  (b, num_head, num_tokens, num_tokens)

context_vec:  torch.Size([1, 4, 768]) # (b, num_tokens, d_out)


As we can see that the size of cached keys and values
cached keys  :  torch.Size([1, 4, 1, 768])  (b, num_tokens, num_head, head_dim)

cached values :  torch.Size([1, 4, 1, 768])  (b, num_tokens, num_head, head_dim)

Model is cached the prompt because our prompt has 4 tokens


##############################################################
##############################################################

Now We have generate the first next token from the prompt and give that token to the model 
 
model(last_token)

queries :  torch.Size([1, 1, 768])  # (b, num_tokens, d_out)

keys  :  torch.Size([1, 1, 768])  # (b, num_tokens, d_out)

values :  torch.Size([1, 1, 768])  # (b, num_tokens, d_out)

now we can see that the num_tokens=1 instead of 5


cached keys  :  torch.Size([1, 5, 1, 768])

cached values :  torch.Size([1, 5, 1, 768])

ealier it has cached 4 tokens and we are appending last 1 so total 5 tokens

we are taking the attn scores of that last one with all other and with itself 

attn scores :  torch.Size([1, 1, 1, 5]) 


context_vec:  torch.Size([1, 1, 768])

context vector with respect to that token, because that what we needed to predict the next word

